In [3]:
import joblib
model = joblib.load("/kaggle/input/adaboost/other/default/1/adaboost_model.pkl")


In [2]:
import pandas as pd 
new_data=pd.read_excel("/kaggle/input/data-ra-final/data_final_Ra.xlsx")


In [3]:
import sys
sys.path.append("/kaggle/input/util-density/utils_density") 
from utils import *

2025-05-21 07:43:07.296039: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747813387.597246      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747813387.697145      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [26]:
data_original_density=pd.read_excel("/kaggle/input/data-original-density/Data Steel by SLM_30nov23.xlsx")
data_original_density.describe()

,Laser Power (W),Laser Speed (mm/s),Hatch Spacing (µm),Layer thickness (µm),Density (%)
count,274.000000,274.000000,268.000000,274.000000,274.000000
mean,244.364964,939.261177,98.432743,52.372263,97.340305
std,144.936828,578.525618,27.364435,62.279884,4.901538
min,70.000000,60.000000,0.975000,20.000000,50.570000
25%,175.000000,600.000000,80.000000,30.000000,97.469763
50%,200.000000,800.000000,100.000000,30.000000,98.684422
75%,255.000000,1200.000000,120.000000,50.000000,99.527500
max,900.000000,3400.000000,240.000000,300.000000,99.999300


In [35]:
data_original_density = data_original_density[data_original_density['Density (%)']>=80]

/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)


In [36]:
data_original_density.describe()

,Laser Power (W),Laser Speed (mm/s),Hatch Spacing (µm),Layer thickness (µm),Density (%)
count,271.000000,271.000000,265.000000,271.000000,271.000000
mean,245.225092,938.109086,98.226321,51.439114,97.713777
std,145.358389,577.570796,27.390104,60.775505,3.219884
min,70.000000,60.000000,0.975000,20.000000,80.570000
25%,175.000000,600.000000,80.000000,30.000000,97.529373
50%,200.000000,800.000000,100.000000,30.000000,98.697924
75%,257.500000,1200.000000,120.000000,50.000000,99.535000
max,900.000000,3400.000000,240.000000,300.000000,99.999300


In [17]:
new_data.describe()

,p,v,h,t,Ra
count,4291.000000,4291.000000,4291.000000,4291.000000,4291.000000
mean,189.960889,787.119186,95.517062,34.063063,9.712378
std,78.865353,244.324573,24.742655,13.473757,4.416855
min,44.336022,71.597240,36.687026,16.385326,1.101585
25%,144.952529,604.210260,79.751008,22.907884,6.705813
50%,172.675031,794.509159,91.287853,30.016912,9.519962
75%,238.067274,985.899334,110.962289,43.105115,11.256840
max,393.167844,1555.213650,200.054491,68.824275,33.281677


In [37]:
import pandas as pd 
new_data=pd.read_excel("/kaggle/input/data-ra-final/data_final_Ra.xlsx")
ved = new_data.iloc[:, 0] / (new_data.iloc[:, 1] * (new_data.iloc[:, 2] / 1000) * (new_data.iloc[:, 3] / 1000))

led = new_data.iloc[:, 0] / new_data.iloc[:, 1]
x_new = np.column_stack((ved, led))


# 2. Transformer avec le même RBFSampler
x_new_rbf = RBFSampler(gamma='scale', n_components=10, random_state=42).fit_transform(x_new)

# 3. Prédire
y_new_norm = model.predict(x_new_rbf).reshape(-1, 1)

scale_y = MinMaxScaler().fit(data_original_density.iloc[:, [4]])

y_new = scale_y.inverse_transform(y_new_norm)

In [38]:
new_data["density"] = y_new

In [43]:
new_data.to_excel("data_final_Ra_Density.xlsx",index=False)

remplissage Ra

In [12]:
import numpy as np
import pandas as pd
data_gmm_density=np.load("/kaggle/input/dataset-denisty-gmm/gmm_dataset_v2.npy")

In [13]:
data_density=pd.read_excel("/kaggle/input/data-original-density/Data Steel by SLM_30nov23.xlsx")
data_density.columns=['p','v','h','t','density']
data_density.head()

,p,v,h,t,density
0,200.0,750.0,90.0,40.0,99.84
1,200.0,1000.0,90.0,40.0,99.75
2,200.0,1250.0,90.0,40.0,99.59
3,200.0,1500.0,90.0,40.0,97.59
4,200.0,1750.0,90.0,40.0,95.00


In [14]:
data_gmm_density = pd.DataFrame(data_gmm_density, columns=data_density.columns)

In [15]:
data_combined = pd.concat([data_density, data_gmm_density], ignore_index=True)

print(data_combined.describe())
data_combined = data_combined.dropna()

                 p            v            h            t      density
count  8274.000000  8274.000000  8268.000000  8274.000000  8274.000000
mean    198.386287  1333.176119    93.112193    37.789841    91.585989
std      62.015523   667.776084    33.207389    16.400174     5.582772
min      70.000000    60.000000     0.975000    20.000000    50.570000
25%     150.099409   802.701923    60.271525    29.936356    87.015744
50%     199.608910  1303.065257    90.106485    30.288264    92.260237
75%     249.138481  1992.585100   119.930300    49.733484    96.359214
max     900.000000  3400.000000   240.000000   300.000000    99.999300


In [17]:
data_combined = data_combined[data_combined['density']>=80]

In [19]:
data_combined.describe()

,p,v,h,t,density
count,8265.000000,8265.000000,8265.000000,8265.000000,8265.000000
mean,198.372430,1333.363485,93.103643,37.754767,91.590695
std,61.980650,667.762212,33.208804,16.149133,5.553269
min,70.000000,60.000000,0.975000,20.000000,80.570000
25%,150.099625,802.705069,60.271219,29.936037,87.015801
50%,199.607472,1302.419246,90.104647,30.287810,92.258739
75%,249.004038,1992.695698,119.930101,49.732017,96.357701
max,900.000000,3400.000000,240.000000,300.000000,99.999300


In [20]:
def box_cox_fun(data, lambda_val=0.5):
    return ((data + 1e-5) ** lambda_val - 1) / lambda_val

def inverse_box_cox(data, lambda_val=0.5):
    if lambda_val == 0:
        return np.exp(data)
    else:
        return (data * lambda_val + 1) ** (1 / lambda_val) - 1e-5


In [21]:
from sklearn.preprocessing import RobustScaler
X=data_combined[['p','v','h','t']].values
scaler=RobustScaler()
X_scaled_combined =scaler.fit_transform(X)

In [22]:

import joblib
from tensorflow.keras.models import load_model

rf = joblib.load("/kaggle/input/models/other/default/1/rf.joblib")
svr = joblib.load("/kaggle/input/models/other/default/1/best_svr.joblib")
xgb = joblib.load("/kaggle/input/models/other/default/1/xgb_best_model.joblib")
mlp = load_model("/kaggle/input/models/other/default/1/best_model.h5", compile=False)
cat = joblib.load("/kaggle/input/cat_boost/other/default/1/cat_model.joblib")
base_models = {
    "rf": rf,
    "svr": svr,
    "xgb": xgb,
    "mlp": mlp,
    "cat": cat
}

selected_base_names = ["cat","svr", "mlp", "xgb"]

def get_preds(name, model, X):
    pred = model.predict(X)
    return pred.squeeze() if name == "mlp" else pred

X_stack_new = np.column_stack([
    get_preds(name, base_models[name], X_scaled_combined)
    for name in selected_base_names
])

meta_model = joblib.load("/kaggle/input/meta_model/other/default/1/best_meta_model.joblib")
y_pred_bc = meta_model.predict(X_stack_new)
y_pred = inverse_box_cox(y_pred_bc)
data_combined['Ra']=y_pred

259/259 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


In [39]:
new_data.describe()

,p,v,h,t,Ra,density
count,4291.000000,4291.000000,4291.000000,4291.000000,4291.000000,4291.000000
mean,189.960889,787.119186,95.517062,34.063063,9.712378,97.588453
std,78.865353,244.324573,24.742655,13.473757,4.416855,2.608616
min,44.336022,71.597240,36.687026,16.385326,1.101585,86.348767
25%,144.952529,604.210260,79.751008,22.907884,6.705813,96.918262
50%,172.675031,794.509159,91.287853,30.016912,9.519962,98.380991
75%,238.067274,985.899334,110.962289,43.105115,11.256840,99.200759
max,393.167844,1555.213650,200.054491,68.824275,33.281677,99.988532


In [24]:
data_combined.describe()


,p,v,h,t,density,Ra
count,8265.000000,8265.000000,8265.000000,8265.000000,8265.000000,8265.000000
mean,198.372430,1333.363485,93.103643,37.754767,91.590695,9.768489
std,61.980650,667.762212,33.208804,16.149133,5.553269,2.764523
min,70.000000,60.000000,0.975000,20.000000,80.570000,4.584858
25%,150.099625,802.705069,60.271219,29.936037,87.015801,6.888441
50%,199.607472,1302.419246,90.104647,30.287810,92.258739,10.741904
75%,249.004038,1992.695698,119.930101,49.732017,96.357701,11.614452
max,900.000000,3400.000000,240.000000,300.000000,99.999300,23.861582


In [25]:
data_combined.to_excel("data_fianl_density_Ra.xlsx")